# 04 · EOC: proximity, capacity, and a briefing

**Atlas source reading copy · browser-network**

[Course guide](index.ipynb) · [Pinned original](https://github.com/jltobias/JupyterLite-WorldView/blob/8e8feea094d489192b8a101069d32fa1328c764f/content/04_eoc_response.ipynb)



Remote services, tiles, CDN assets or package retrieval are subject to network/CORS availability.

**Provenance:** `content/04_eoc_response.ipynb` at `8e8feea094d489192b8a101069d32fa1328c764f`; original SHA-256 `7a2fe28a8566c4b67470ec84527ff5ca82d6811940a42782885bc03aa0369645`. Source-specific license/notices retained under notices; data and media have separate terms.

**Verification:** imported by static inspection; saved upstream outputs cleared. Runtime compatibility is separate from executed verification. See the course guide for dependencies, excluded assets, aliases and original credits.


# 04 · EOC: proximity, capacity, and a briefing

**Mission:** identify facilities near a hypothetical incident and quantify available exercise capacity. Allow 35 minutes. Prerequisite: Lab 03.

**Run:** choose the Python (Pyodide) kernel in JupyterLite, then Run → Run All Cells. First use downloads Matplotlib. All training data are **SYNTHETIC** unless explicitly labeled LIVE. Downloads appear below and in `exports/`; the browser filesystem is separate from your computer. Each lab runs independently.

In [ ]:
import sys
if sys.platform == "emscripten":
    import micropip
    await micropip.install("matplotlib")
import json, math
from pathlib import Path
from IPython.display import display, HTML
import matplotlib.pyplot as plt
from worldview_lab import (load, export, map_layer, chart_style, haversine_km,
    wilson_interval, shortest_times, freshness, normalize_quakes,
    validate_geojson, brief_request, validate_brief)
chart_style()

## Scenario: a hypothetical service interruption

A fictional incident at `(-77.10, 38.83)` triggers a **20 km screening radius**. This radius is an exercise parameter, not a forecast damage footprint. Distance alone cannot establish impact, safety, travel time, or a recommended dispatch.

In [ ]:
INCIDENT=(-77.10,38.83);RADIUS_KM=20
facilities=load('facilities.geojson');screen=[]
for f in facilities['features']:
    p=f['properties'];lon,lat=f['geometry']['coordinates']
    d=haversine_km(*INCIDENT,lon,lat)
    screen.append({'id':p['id'],'name':p['name'],'distance_km':round(d,2),'open':p['open'],
                   'capacity':p['capacity'],'inside':d<=RADIUS_KM})
display(screen)
eligible=[r for r in screen if r['inside'] and r['open']]
print('Open capacity inside screening radius:',sum(r['capacity'] for r in eligible),'exercise slots')

In [ ]:
fig,ax=plt.subplots()
ax.barh([r['id'] for r in screen],[r['distance_km'] for r in screen],color=['#148f97' if r['open'] else '#999999' for r in screen])
ax.axvline(RADIUS_KM,color='#d45b63',linestyle='--',label='Screening radius')
ax.set(xlabel='Great-circle distance (km)',title='SYNTHETIC EOC proximity screening');ax.legend();plt.show()
map_layer(facilities,value='capacity',title='SYNTHETIC response facilities')

In [ ]:
evidence={'classification':'SYNTHETIC EXERCISE','scenario_clock':'2026-09-14T12:00:00Z',
 'evidence':[{'id':'EOC-01','metric':'open_capacity_within_radius','value':sum(r['capacity'] for r in eligible),'unit':'exercise slots','radius_km':RADIUS_KM},
             {'id':'EOC-02','metric':'facility_screen','rows':screen}],
 'limitations':['Spherical distance, not a road route','Capacity and open status are fictional','No hazard model or facility safety assessment']}
export('eoc-evidence.json',evidence)

## Sensitivity and review

Change the radius to 10 km and 30 km. Close the central facility. Identify whether the conclusion changes because of geography or status. **Checkpoint:** a closed facility contributes zero available capacity even if it is closest.

Draft a three-sentence handoff: what was calculated, what remains unknown, and who should verify status. Use the exported evidence with Lab 08 for an optional Astra draft. A map should inform the review, not silently become a dispatch authority.

## Sources and attribution

[WHO EOC framework](https://www.who.int/publications/i/item/framework-for-a-public-health-emergency-operations-centre). This proximity exercise is an original teaching scenario, not a WHO operational procedure.

World View inspiration: [Kevin (Khoa) To, MIT](https://github.com/kevtoe/worldview). Original lab code, figures, and synthetic fixtures: JupyterLite WorldView contributors, MIT. See the [book references](https://jltobias.github.io/JupyterLite-WorldView/book/references.html) and repository notices for dependency and service terms.